# EDA on Online Retail Sales

**Oasis Infobyte — Data Analytics Level 1, Task 1**

This notebook analyses the supplied Online Retail dataset: inspection, cleaning, descriptive statistics, monthly/quarterly trends, market and product analysis, correlation analysis, and recommendations.

**Limitation:** the dataset has no age or gender fields, so those analyses are not fabricated; country/market analysis is used instead.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
BASE=Path.cwd(); DATA=BASE/'data'/'raw'/'online_retail.csv'; CLEAN=BASE/'data'/'cleaned'/'online_retail_cleaned.csv'; OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)
df_raw=pd.read_csv(DATA); display(df_raw.head())

## 1. Initial inspection

In [ ]:
print('Shape:',df_raw.shape)
display(df_raw.dtypes.rename('dtype').to_frame())
display(df_raw.isna().sum().sort_values(ascending=False).rename('missing_values').to_frame())
print('Duplicate rows:',df_raw.duplicated().sum())
display(df_raw.describe(include='all').T)

## 2. Data cleaning

Convert dates, trim text, remove exact duplicates, calculate Revenue, identify cancelled invoices, and exclude cancellations/returns and non-positive sales from sales-performance analysis.

In [ ]:
df=df_raw.copy(); before={'rows':len(df),'duplicates':int(df.duplicated().sum()),'missing_cells':int(df.isna().sum().sum())}
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'],errors='coerce')
for c in ['Description','Country','StockCode','InvoiceNo']: df[c]=df[c].astype('string').str.strip()
df=df.drop_duplicates().copy(); df['is_cancelled']=df['InvoiceNo'].str.upper().str.startswith('C',na=False); df['Revenue']=df['Quantity']*df['UnitPrice']
sales_df=df[(~df['is_cancelled'])&(df['Quantity']>0)&(df['UnitPrice']>0)&df['InvoiceDate'].notna()].copy()
after={'rows':len(sales_df),'duplicates':0,'missing_cells':int(sales_df.isna().sum().sum())}; display(pd.DataFrame({'Before':before,'After':after})); sales_df.to_csv(CLEAN,index=False)

## 3. Descriptive statistics

In [ ]:
display(sales_df[['Quantity','UnitPrice','Revenue']].describe().T)
print('Unique invoices:',sales_df.InvoiceNo.nunique()); print('Unique products:',sales_df.StockCode.nunique()); print('Unique countries:',sales_df.Country.nunique())

## 4. Monthly and quarterly sales trends

In [ ]:
sales_df['Month']=sales_df.InvoiceDate.dt.to_period('M').astype(str); sales_df['Quarter']=sales_df.InvoiceDate.dt.to_period('Q').astype(str)
monthly=sales_df.groupby('Month').Revenue.sum(); quarterly=sales_df.groupby('Quarter').Revenue.sum()
fig,ax=plt.subplots(figsize=(12,5)); monthly.plot(marker='o',ax=ax); ax.set_title('Monthly Revenue Trend'); ax.set_xlabel('Month'); ax.set_ylabel('Revenue'); plt.tight_layout(); plt.savefig(OUT/'monthly_revenue_trend.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(11,5)); quarterly.plot(marker='o',ax=ax); ax.set_title('Quarterly Revenue Trend'); ax.set_xlabel('Quarter'); ax.set_ylabel('Revenue'); plt.tight_layout(); plt.savefig(OUT/'quarterly_revenue_trend.png',dpi=160); plt.show()
print('Peak month:',monthly.idxmax(),monthly.max()); print('Peak quarter:',quarterly.idxmax(),quarterly.max())

### Key visualisations

![Monthly Revenue Trend](outputs/monthly_revenue_trend.png)

![Quarterly Revenue Trend](outputs/quarterly_revenue_trend.png)

## 5. Market analysis

Age and gender are absent from the supplied data. Country analysis is therefore used as the available market/customer dimension.

In [ ]:
country_revenue=sales_df.groupby('Country').Revenue.sum().sort_values(ascending=False); country_orders=sales_df.groupby('Country').InvoiceNo.nunique().sort_values(ascending=False)
fig,ax=plt.subplots(figsize=(10,6)); country_revenue.head(10).sort_values().plot(kind='barh',ax=ax); ax.set_title('Top 10 Countries by Revenue'); ax.set_xlabel('Revenue'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_revenue.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(10,6)); country_orders.head(10).sort_values().plot(kind='barh',ax=ax); ax.set_title('Top 10 Countries by Number of Orders'); ax.set_xlabel('Unique invoices'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_orders.png',dpi=160); plt.show()

## 6. Product analysis

In [ ]:
top_products=sales_df.groupby('Description').Quantity.sum().sort_values(ascending=False).head(10); display(top_products.to_frame('units_sold'))
fig,ax=plt.subplots(figsize=(10,6)); top_products.sort_values().plot(kind='barh',ax=ax); ax.set_title('Top 10 Products by Units Sold'); ax.set_xlabel('Units'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_units.png',dpi=160); plt.show()

### Top 10 products by units sold

![Top 10 Products by Units Sold](outputs/top_10_products_by_units_sold.jpg)

## 7. Correlation heatmap

In [ ]:
plt.figure(figsize=(8,6)); sns.heatmap(sales_df[['Quantity','UnitPrice','Revenue']].corr(),annot=True,fmt='.2f',cmap='coolwarm'); plt.title('Correlation Matrix'); plt.tight_layout(); plt.savefig(OUT/'correlation_heatmap.png',dpi=160); plt.show()

### Correlation matrix

![Correlation Matrix](outputs/correlation_matrix.jpg)

## 8. Additional insight and recommendations

Revenue concentration is assessed by country, alongside customer-level revenue where CustomerID exists.

Recommendations: plan inventory and promotions around peak periods; protect availability of high-volume products and use bundles; focus retention on major revenue markets while testing smaller markets selectively; improve CustomerID capture for stronger retention and lifetime-value analysis.

In [ ]:
total=sales_df.Revenue.sum(); uk_share=sales_df.loc[sales_df.Country.eq('United Kingdom'),'Revenue'].sum()/total; top10_share=country_revenue.head(10).sum()/total
print(f'UK revenue share: {uk_share:.1%}'); print(f'Top-10-country revenue share: {top10_share:.1%}')
customer_summary=sales_df.dropna(subset=['CustomerID']).groupby('CustomerID').agg(Orders=('InvoiceNo','nunique'),Revenue=('Revenue','sum'),Units=('Quantity','sum')).sort_values('Revenue',ascending=False); display(customer_summary.head(10))

## 9. Conclusion

The cleaned dataset supports transaction-level retail sales analysis. The main decision areas are temporal demand planning, product availability, market concentration, and improving customer-level data completeness.